# Four-Body Zöllner Electrogravitation — Regularized Reference

Four-body Weber electrodynamics with Zöllner electrogravitational extension.
Four runs with mismatch parameter **`a ∈ {0, 0.02, 0.05, 0.10}`**, all using
the regularized integrator with **identical initial conditions**.

**Highlights:**
- Analytically-derived ICs: speed from target energy ratio η = KE / |PE₀|
- Square geometry, alternating (+,−,+,−) charges, tangential CCW momenta
- All ṙᵢⱼ = 0 at t = 0 → Weber potential = Coulomb at t = 0 (closed-form IC)
- Relative-coordinate analysis: rᵢⱼ(t), (r, ṙ) phase portraits, pair energies
- Zöllner residuals scale linearly with `a` (quantitatively verified)

In [ ]:
using WeberElectrodynamics
using LinearAlgebra
using Plots
using Printf

## 1. Theory & Initial Condition Derivation

### Weber Hamiltonian with Zöllner

$$H = \sum_i \frac{|\mathbf{p}_i|^2}{2m_i} + \sum_{i<j} \frac{\kappa_{ij}\, q_i q_j}{r_{ij}}\!\left(1 - \frac{\dot{r}_{ij}^2}{2c^2}\right)$$

where $\kappa_{ij} = 1 + a$ for unlike-sign pairs, $\kappa_{ij} = 1$ otherwise.

### Square Geometry (alternating charges)

```
        2(−Q)  North
       ↙  ↑
1(+Q) →        ← 3(+Q)
       ↘  ↓
        4(−Q)  South
```

Positions at radius $R$ (cardinal points):
$$q_1=(R,0),\quad q_2=(0,R),\quad q_3=(-R,0),\quad q_4=(0,-R)$$

Tangential (CCW) momenta (mass $m$, speed $v$):
$$p_1=mv(0,1),\quad p_2=mv(-1,0),\quad p_3=mv(0,-1),\quad p_4=mv(1,0)$$

**Properties at $t=0$:**
- $\sum_i \mathbf{p}_i = 0$ exactly (zero total momentum by symmetry)
- $\dot{r}_{ij} = \hat{r}_{ij}\cdot(\mathbf{v}_i - \mathbf{v}_j) = 0$ for all pairs (velocity ⊥ separation)
  → Weber potential reduces to **pure Coulomb** at $t=0$

### Pair Structure

| Pair   | Product | Distance | κ     |
|--------|---------|----------|-------|
| (1,2)  | −Q²     | R√2      | 1+a   |
| (1,3)  | +Q²     | 2R       | 1     |
| (1,4)  | −Q²     | R√2      | 1+a   |
| (2,3)  | −Q²     | R√2      | 1+a   |
| (2,4)  | +Q²     | 2R       | 1     |
| (3,4)  | −Q²     | R√2      | 1+a   |

**4 unlike-sign adjacent pairs** receive the Zöllner boost κ = 1+a.
**2 like-sign diagonal pairs** are unaffected (repulsive structural restoring force).

### Closed-Form Speed from Energy Target

At $t=0$ with all $\dot{r}_{ij}=0$ and $a=0$:
$$|{\rm PE}_0| = \frac{Q^2}{R}(2\sqrt{2}-1)$$

Choosing kinetic fraction $\eta = {\rm KE}/|{\rm PE}_0|$:
$$v = \sqrt{\frac{\eta\,|{\rm PE}_0|}{2m}}$$

For $\eta < \eta_{\rm circ} \approx 0.82$ particles have sub-circular speed
→ spiral inward → **close encounters → regularization is essential**.

In [ ]:
# ── Physical parameters ──────────────────────────────────────────────────────
n_particles = 4
dims        = 2
Q           = sqrt(0.1)               # charge magnitude
masses      = ones(4)
charges     = [Q, -Q, Q, -Q]          # alternating (+,−,+,−)
c           = 4.0

# ── IC geometry ──────────────────────────────────────────────────────────────
R   = 0.8     # orbital radius
eta = 0.70    # target KE / |PE₀|

# |PE₀| at t=0: 4 unlike adjacent pairs at R√2 + 2 like diagonal pairs at 2R
# PE₀ = 4·(−Q²)/(R√2) + 2·Q²/(2R) = Q²/R·(1 − 2√2)
PE_abs_0  = Q^2 / R * (2*sqrt(2) - 1)
KE_target = eta * PE_abs_0             # = 2mv² (4 particles, m=1)
v         = sqrt(KE_target / 2)       # since KE = 2·m·v²

# ── Initial positions (cardinal points of square) ────────────────────────────
q0 = Float64[
     R,  0.0,   # 1 (+Q) East
    0.0,  R,    # 2 (−Q) North
    -R,  0.0,   # 3 (+Q) West
    0.0, -R,    # 4 (−Q) South
]

# ── Initial momenta (tangential, CCW) ────────────────────────────────────────
p0 = Float64[
    0.0,  v,    # 1: North
     -v, 0.0,   # 2: West
    0.0, -v,    # 3: South
      v, 0.0,   # 4: East
]

# ── Pair lists ────────────────────────────────────────────────────────────────
pairs_list   = [(i,j) for i in 1:n_particles for j in (i+1):n_particles]
unlike_pairs = [(i,j) for (i,j) in pairs_list if charges[i]*charges[j] < 0]
like_pairs   = [(i,j) for (i,j) in pairs_list if charges[i]*charges[j] > 0]

# ── Integration & regularization parameters ───────────────────────────────────
tspan    = (0.0, 30.0)
dt       = 0.001
reg_r_on = 0.30    # regularize when rij < reg_r_on
reg_r_off = 0.50
n_pairs  = n_particles * (n_particles - 1) ÷ 2

# ── Colour palette (consistent across all plots) ─────────────────────────────
a_values   = [0.0,  0.02,  0.05,  0.10]
run_labels = ["a=0 (Weber)", "a=0.02", "a=0.05", "a=0.10"]
a_colors   = [:black, :steelblue, :darkorange, :firebrick]

# ── System construction ──────────────────────────────────────────────────────
system = WeberSystem(n_particles, dims)

@printf("Parameters: Q=%.4f  m=1  R=%.2f  c=%.1f  eta=%.2f\n", Q, R, c, eta)
@printf("IC speed:   v=%.6f  (circular orbit v_circ ≈ 0.306)\n", v)
@printf("|PE₀|=%.6f  KE_target=%.6f\n", PE_abs_0, KE_target)

In [ ]:
# Verify initial conditions
kappas_ref = ones(n_pairs)   # a=0 reference
H0_ref     = system.hamiltonian_compiled(q0, p0,
                 vcat(masses, charges, [c], kappas_ref))
KE_actual  = sum(p0.^2) / 2
PE_actual  = H0_ref - KE_actual
eta_actual = KE_actual / abs(PE_actual)
P_total    = [sum(p0[1:2:end]), sum(p0[2:2:end])]
Lz         = sum(q0[1:2:end] .* p0[2:2:end] .- q0[2:2:end] .* p0[1:2:end])

println("Initial condition verification (a=0 reference):")
@printf("  KE        = %.8f  (target %.8f)\n", KE_actual, KE_target)
@printf("  PE        = %.8f\n", PE_actual)
@printf("  H₀        = %.8f  (bound: %s)\n", H0_ref, H0_ref < 0 ? "yes ✓" : "no ✗")
@printf("  η         = %.4f  (target %.2f)\n", eta_actual, eta)
@printf("  |P_total| = %.2e  (should be ≈ 0)\n", norm(P_total))
@printf("  Lz        = %.6f  (= 4mvR = %.6f)\n", Lz, 4*1.0*v*R)

println("\nRadial velocities at t=0 (all should be ≈ 0):")
for (i,j) in pairs_list
    si  = (i-1)*dims
    sj  = (j-1)*dims
    rij = q0[si+1:si+dims] - q0[sj+1:sj+dims]
    vij = p0[si+1:si+dims]./masses[i] - p0[sj+1:sj+dims]./masses[j]
    rdot = dot(rij, vij) / norm(rij)
    sign_str = charges[i]*charges[j] < 0 ? "unlike" : "like  "
    @printf("  pair (%d,%d) %s  ṙ = %.2e\n", i, j, sign_str, rdot)
end

In [ ]:
# Visualise initial configuration: positions, charges, velocity arrows
q_x    = [q0[2k-1] for k in 1:n_particles]
q_y    = [q0[2k]   for k in 1:n_particles]
vx     = [p0[2k-1] for k in 1:n_particles]   # mass=1 so v=p
vy     = [p0[2k]   for k in 1:n_particles]
q_syms = ["+Q", "−Q", "+Q", "−Q"]
q_cols = [:firebrick, :steelblue, :firebrick, :steelblue]

p_ic = scatter(q_x, q_y;
    markersize        = 20,
    markershape       = :circle,
    markercolor       = q_cols,
    markerstrokewidth = 2,
    label             = "",
    aspect_ratio      = :equal,
    xlims = (-1.4, 1.4), ylims = (-1.4, 1.4),
    title  = "Initial Configuration (t=0)",
    xlabel = "x", ylabel = "y",
    framestyle = :box, grid = true, gridalpha = 0.25,
    dpi = 200, size = (650, 650),
)

# Charge labels
for k in 1:n_particles
    annotate!(p_ic, q_x[k], q_y[k], text(q_syms[k], :white, :center, 10, :bold))
    offsets = [(0.18,0.18),(-0.22,0.18),(-0.22,-0.20),(0.18,-0.20)]
    annotate!(p_ic, q_x[k]+offsets[k][1], q_y[k]+offsets[k][2],
              text("P$k", :black, :center, 9))
end

# Velocity arrows (lines from each particle in velocity direction)
arrow_scale = 1.3
for k in 1:n_particles
    x0, y0 = q_x[k], q_y[k]
    dx, dy  = arrow_scale*vx[k], arrow_scale*vy[k]
    plot!(p_ic, [x0, x0+dx], [y0, y0+dy];
        arrow = true, color = :gray40, linewidth = 2.0,
        label = k == 1 ? "velocity (×$(arrow_scale))" : "")
end

# Square outline (connecting adjacent particles)
sq_order = [1, 2, 3, 4, 1]
plot!(p_ic, q_x[sq_order], q_y[sq_order];
    linestyle = :dot, color = :gray, linewidth = 1, label = "")

# Pair labels (unlike/like)
annotate!(p_ic,  0.0,  1.2, text("← like (1,3) →", :gray50, :center, 8))
annotate!(p_ic,  0.0, -1.2, text("← like (2,4) →", :gray50, :center, 8))

p_ic

In [ ]:
# Build four WeberProblems — identical ICs, only zollner_a differs
reg_opts = RegularizationOptions(
    enabled       = true,
    backend       = :lifted_pair,
    r_on          = reg_r_on,
    r_off         = reg_r_off,
    max_substeps  = 512,
    chain_enabled = true,
    warn_on_fallback = false,
)

probs = [
    WeberProblem(system, tspan, q0, p0;
        masses  = masses,
        charges = charges,
        c       = c,
        dt      = dt,
        regularization = reg_opts,
        zollner = ZollnerOptions(enabled = (a > 0.0), a = a),
    )
    for a in a_values
]

# Print κ table
println("κ values per run:")
@printf("  %-18s  %s\n", "Run", "κ₁₂  κ₁₃  κ₁₄  κ₂₃  κ₂₄  κ₃₄")
for (prob, label) in zip(probs, run_labels)
    kstr = join([@sprintf("%.4g", k) for k in prob.kappas], "  ")
    @printf("  %-18s  %s\n", label, kstr)
end

## 3. Coupling Constants κ

For the alternating charge layout `[+Q, −Q, +Q, −Q]`:

- **4 unlike-sign adjacent pairs** `(1,2), (1,4), (2,3), (3,4)` → `κ = 1 + a`
  These are the attractive pairs that drive close encounters.
- **2 like-sign diagonal pairs** `(1,3), (2,4)` → `κ = 1` regardless of `a`
  These are repulsive and act as a structural restoring force.

The Zöllner effect is purely additive on the unlike-sign pairs.
At `a = 0.10`, the adjacent-pair attraction is 10% stronger than standard Weber,
making the system more deeply bound and altering the orbital dynamics.

In [ ]:
# Integrate all four runs
println("Integrating $(length(probs)) runs × $(Int((tspan[2]-tspan[1])/dt)) steps each …")
sols = [solve(prob) for prob in probs]

println("\nIntegration results:")
@printf("  %-18s  %-9s  %-8s  %-12s  %-12s  %-12s\n",
    "Run", "Retcode", "Points", "pair_steps", "chain_steps", "min_sep")
for (sol, label) in zip(sols, run_labels)
    rd = sol.regularization
    @printf("  %-18s  %-9s  %-8d  %-12d  %-12d  %-12.4e\n",
        label, sol.retcode, length(sol.t),
        rd.pair_steps, rd.chain_steps, rd.min_encounter_distance)
end

## 4. Regularization Activity

The Levi-Civita / KS regularization activates when any pair separation
drops below `r_on = 0.30`. Since η = 0.70 is below the circular-orbit
threshold (η_circ ≈ 0.82), particles have insufficient centrifugal support
and spiral inward, triggering many regularized sub-steps.

- **pair_steps**: timesteps where ≥ 1 pair was being regularized
- **chain_steps**: timesteps with simultaneous multi-pair close encounters
- **min_encounter_distance**: smallest separation reached across the full run

As `a` increases, the unlike-sign pairs attract more strongly → closer
encounters → more regularization work.

In [ ]:
# 2D trajectory plots for all four runs (2×2 layout)
traj_plots = []
for (sol, label) in zip(sols, run_labels)
    traj = compute_trajectory_data(sol, n_particles, dims; stride=1)
    p = plot_trajectories(traj)
    title!(p, "Trajectories — $label")
    push!(traj_plots, p)
end

plot(traj_plots..., layout=(2,2), size=(2100, 2100),
    plot_title="Regularized 2D Trajectories — All Four Runs")

In [ ]:
# Overlay: a=0 (Weber) vs a=0.10 (max Zöllner) — clearest visual contrast
plot_weber_vs_zollner(sols[1], sols[4];
    labels = [run_labels[1], run_labels[4]])

In [ ]:
# Helper: extract separation distance rᵢⱼ(t) from solution
function pair_separation(sol::WeberSolution, i::Int, j::Int, dims::Int)
    [begin
        qi = sol.q[k][(i-1)*dims+1 : i*dims]
        qj = sol.q[k][(j-1)*dims+1 : j*dims]
        norm(qi - qj)
    end for k in eachindex(sol.t)]
end

# Precompute all 6 pairwise separations for all 4 runs
all_seps = Dict{Tuple{Int,Int}, Vector{Vector{Float64}}}()
for (i,j) in pairs_list
    all_seps[(i,j)] = [pair_separation(sol, i, j, dims) for sol in sols]
end

println("Pairwise separations at t=0 and minimum over full run:")
@printf("  %-8s  %-8s  %-6s  %s\n", "Pair", "Sign", "r₀", "min r per run (a=0, 0.02, 0.05, 0.10)")
for (i,j) in pairs_list
    sn  = charges[i]*charges[j] < 0 ? "unlike" : "like  "
    r0  = all_seps[(i,j)][1][1]
    mins = join([@sprintf("%.4f", minimum(r)) for r in all_seps[(i,j)]], "  ")
    @printf("  (%d,%d)    %-8s  %.4f  %s\n", i, j, sn, r0, mins)
end

In [ ]:
# 6-panel: rᵢⱼ(t) for all pairs, all 4 runs overlaid
sep_panels = []
for (ip, (i,j)) in enumerate(pairs_list)
    sn      = charges[i]*charges[j] < 0 ? "unlike κ=1+a" : "like   κ=1  "
    is_last = ip >= 5
    p = plot(;
        title        = "r_{$i$j}(t)  [$sn]",
        xlabel       = is_last ? "Time t" : "",
        ylabel       = "rᵢⱼ",
        legend       = ip == 1 ? :topright : :none,
        framestyle   = :box, grid = true, gridalpha = 0.2,
        tickfontsize = 8, guidefontsize = 9, titlefontsize = 9,
        dpi = 200, linewidth = 1.3,
    )
    for (k, (sol, label, clr)) in enumerate(zip(sols, run_labels, a_colors))
        plot!(p, sol.t, all_seps[(i,j)][k];
            label = label, color = clr,
            linewidth = k == 1 ? 2.0 : 1.3, alpha = 0.85)
    end
    hline!(p, [reg_r_on]; linestyle = :dash, color = :gray60,
           linewidth = 0.8, label = ip == 1 ? "r_on" : "")
    push!(sep_panels, p)
end

plot(sep_panels..., layout=(3,2), size=(2100, 1575),
    plot_title="Pairwise Separations rᵢⱼ(t) — All 4 Runs")

In [ ]:
# Precompute force timeseries: 4 unlike-sign pairs × 4 runs
println("Computing force timeseries ($(length(unlike_pairs)) pairs × $(length(sols)) runs) …")
forces_matrix = [
    [compute_pair_force_timeseries(
         sol, pair, n_particles, dims, masses, charges, c; stride=1)
     for pair in unlike_pairs]
    for sol in sols
]

# Also like-sign pair (1,3) as control
forces_like13 = [
    compute_pair_force_timeseries(
        sol, (1,3), n_particles, dims, masses, charges, c; stride=1)
    for sol in sols
]

println("Done.\n")
println("Force summary — pair (1,2) [unlike, κ=1+a]:")
@printf("  %-18s  %-12s  %-8s  %-14s\n", "Run", "mean |F|", "κ", "mean Zöll extra")
for (k, label) in enumerate(run_labels)
    fd  = forces_matrix[k][1]   # pair (1,2)
    mze = sum(fd.zollner_extra_magnitude) / length(fd.zollner_extra_magnitude)
    @printf("  %-18s  %-12.4e  %-8.4f  %-14.4e\n", label, fd.stats.mean, fd.kappa, mze)
end

In [ ]:
# Phase portraits (r, ṙ) — one panel per unlike-sign pair, all 4 a-values overlaid
phase_panels = []
for (pi, (i,j)) in enumerate(unlike_pairs)
    p = plot(;
        title        = "Phase portrait — pair ($i,$j)  [κ=1+a]",
        xlabel       = "Separation r",
        ylabel       = "Radial velocity ṙ",
        legend       = pi == 1 ? :topright : :none,
        framestyle   = :box, grid = true, gridalpha = 0.2,
        tickfontsize = 8, guidefontsize = 9, titlefontsize = 9,
        dpi = 200, linewidth = 1.3,
    )
    for (k, (label, clr)) in enumerate(zip(run_labels, a_colors))
        fd = forces_matrix[k][pi]
        ps = fd.phase_space
        plot!(p, ps.separation_distance, ps.radial_velocity;
            label = label, color = clr,
            linewidth = k == 1 ? 2.0 : 1.3, alpha = 0.75)
        scatter!(p, [ps.separation_distance[1]], [ps.radial_velocity[1]];
            marker = :circle, markersize = 5, color = clr, label = "")
    end
    vline!(p, [reg_r_on]; linestyle = :dash, color = :gray60,
           linewidth = 0.8, label = pi == 1 ? "r_on" : "")
    push!(phase_panels, p)
end

plot(phase_panels..., layout=(2,2), size=(2100, 2100),
    plot_title="Phase Portraits (r, ṙ) — Unlike-Sign Pairs, All 4 Runs")

In [ ]:
# Control: phase portrait for like-sign diagonal pair (1,3) [κ=1 for all a]
p_like = plot(;
    title  = "Phase portrait — pair (1,3)  [like-sign, κ=1 always]",
    xlabel = "Separation r",
    ylabel = "Radial velocity ṙ",
    legend = :topright,
    framestyle = :box, grid = true, gridalpha = 0.2,
    dpi = 200, linewidth = 1.3, size = (1050, 787),
)
for (k, (label, clr)) in enumerate(zip(run_labels, a_colors))
    fd = forces_like13[k]
    ps = fd.phase_space
    plot!(p_like, ps.separation_distance, ps.radial_velocity;
        label = label, color = clr, linewidth = k == 1 ? 2.0 : 1.3, alpha = 0.75)
end
vline!(p_like, [reg_r_on]; linestyle = :dash, color = :gray60, linewidth = 0.8,
       label = "r_on")
p_like

In [ ]:
# plot_zollner_phase_space: Weber (a=0) vs Zöllner (a=0.10) for pair (1,2)
plot_zollner_phase_space(forces_matrix[1][1], forces_matrix[4][1];
    labels = [run_labels[1], run_labels[4]])

In [ ]:
# Compute energy timeseries for all four runs
println("Computing energy timeseries …")
energies = [compute_energy_timeseries(sol; stride=1) for sol in sols]
println("Done.\n")

println("Energy conservation summary:")
@printf("  %-18s  %-12s  %-14s  %-14s\n",
    "Run", "H₀", "max drift %", "max local err")
for (en, label) in zip(energies, run_labels)
    s = en.statistics
    @printf("  %-18s  %-12.6f  %-14.4e  %-14.4e\n",
        label, en.total_energy[1],
        s.global_error_percent_max, s.local_error_max)
end

In [ ]:
# Energy timeseries: KE, PE, total, relative error — all four runs
plot(
    [plot_energy(en) for en in energies]...,
    layout=(2,2), size=(2100, 2100),
    plot_title="Energy Conservation — All Four Runs",
)

In [ ]:
# Energy error analysis: local, global drift, Hamiltonian validation
plot(
    [plot_energy_errors(en) for en in energies]...,
    layout=(2,2), size=(2100, 2625),
    plot_title="Energy Error Analysis — All Four Runs",
)

In [ ]:
# Zöllner energy breakdown: per-pair residual + total gravitational residual
pze_05 = plot_zollner_energy(energies[3])   # a=0.05
pze_10 = plot_zollner_energy(energies[4])   # a=0.10

plot(pze_05, pze_10, layout=(1,2), size=(2100, 787),
    plot_title="Zöllner Extra Potential (emergent gravity) — a=0.05 vs a=0.10")

In [ ]:
# Per-pair energy detail for pair (1,2) [unlike-sign, κ=1+a]
# Coulomb term, velocity correction term, total pair potential, radial velocity ṙ
pe12_w = plot_pair_energy(energies[1], (1,2))   # a=0 Weber
pe12_z = plot_pair_energy(energies[4], (1,2))   # a=0.10 Zöllner

plot(pe12_w, pe12_z, layout=(1,2), size=(2100, 787),
    plot_title="Pair (1,2) Energy Detail [unlike-sign κ=1+a]: Weber vs Zöllner a=0.10")

In [ ]:
# Per-pair energy for pair (1,3) [like-sign, κ=1 always — control]
pe13_w = plot_pair_energy(energies[1], (1,3))   # a=0
pe13_z = plot_pair_energy(energies[4], (1,3))   # a=0.10

plot(pe13_w, pe13_z, layout=(1,2), size=(2100, 787),
    plot_title="Pair (1,3) Energy Detail [like-sign κ=1 always]: Weber vs Zöllner a=0.10")

In [ ]:
# Full Weber force decomposition — pair (1,2), a=0 baseline
# Four panels: |F|, components (Fx,Fy), vector form, radial form
plot_pair_forces(forces_matrix[1][1])

In [ ]:
# Full Weber force decomposition — pair (1,2), a=0.05
plot_pair_forces(forces_matrix[3][1])

In [ ]:
# Zöllner extra force vs total — pair (1,2) for a=0.05 and a=0.10
pzfr_05 = plot_zollner_force_residual(forces_matrix[3][1])
pzfr_10 = plot_zollner_force_residual(forces_matrix[4][1])

plot(pzfr_05, pzfr_10, layout=(1,2), size=(2100, 787),
    plot_title="Zöllner Force Residual — Pair (1,2): a=0.05 vs a=0.10")

In [ ]:
# Quantitative Zöllner scaling across all four a values
zollner_max_residuals = Float64[]
mean_zollner_fracs    = Float64[]
energy_drift_percents = Float64[]
min_separations       = Float64[]

for (k, (en, sol)) in enumerate(zip(energies, sols))
    max_res  = maximum(abs.(en.total_zollner_residual))
    fd12     = forces_matrix[k][1]   # pair (1,2)
    eps_eps  = eps(Float64)
    mze_frac = sum(fd12.zollner_extra_magnitude ./
                   max.(fd12.magnitude, eps_eps)) / length(fd12.magnitude) * 100
    push!(zollner_max_residuals, max_res)
    push!(mean_zollner_fracs, mze_frac)
    push!(energy_drift_percents, en.statistics.global_error_percent_max)
    push!(min_separations, sol.regularization.min_encounter_distance)
end

println("Zöllner scaling summary:")
@printf("  %-18s  %-16s  %-16s  %-14s  %-12s\n",
    "Run", "max|ΔE_Zöllner|", "mean Zöll%", "energy drift%", "min sep")
for (k, label) in enumerate(run_labels)
    @printf("  %-18s  %-16.4e  %-16.4f  %-14.4e  %-12.4e\n",
        label, zollner_max_residuals[k], mean_zollner_fracs[k],
        energy_drift_percents[k], min_separations[k])
end

In [ ]:
# Three-panel: Zöllner residuals scale linearly with a
pd = (framestyle=:box, grid=true, gridalpha=0.2,
      tickfontsize=10, guidefontsize=11, titlefontsize=12,
      linewidth=2.0, markersize=7, dpi=200)

a_ref    = collect(range(0.0, 0.10; length=50))
slope_e  = a_values[end] > 0 ? zollner_max_residuals[end]/a_values[end] : 0.0
slope_f  = a_values[end] > 0 ? mean_zollner_fracs[end]/a_values[end]    : 0.0

p_res = plot(a_values, zollner_max_residuals;
    title="max |ΔE_Zöllner| vs a",
    xlabel="a", ylabel="max |Σ (κ−1)·qᵢqⱼ/r·(1−ṙ²/2c²)|",
    marker=:circle, color=:firebrick, label="measured", pd...)
plot!(p_res, a_ref, slope_e .* a_ref;
    linestyle=:dash, color=:gray, label="linear ref")

p_frac = plot(a_values, mean_zollner_fracs;
    title="Mean Zöllner Force Fraction vs a",
    xlabel="a", ylabel="|(κ−1)F_Coulomb| / |F_total|  (%)",
    marker=:diamond, color=:darkorange, label="measured", pd...)
plot!(p_frac, a_ref, slope_f .* a_ref;
    linestyle=:dash, color=:gray, label="linear ref")

p_sep = plot(a_values, min_separations;
    title="Min Encounter Distance vs a",
    xlabel="a", ylabel="min rᵢⱼ (full run)",
    marker=:square, color=:steelblue, label="min sep", pd...)

plot(p_res, p_frac, p_sep, layout=(1,3), size=(3150, 787),
    plot_title="Zöllner Scaling with Mismatch Parameter a")

In [ ]:
# Linear and angular momentum — all four runs (2×2)
mom_plots = []
for (sol, label) in zip(sols, run_labels)
    m = compute_momentum_timeseries(sol; stride=1)
    p = plot_momentum_errors(m)
    title!(p.subplots[1], "Momentum — $label")
    push!(mom_plots, p)
end

plot(mom_plots..., layout=(2,2), size=(2100, 2100),
    plot_title="Momentum Conservation — All Four Runs")

In [ ]:
# Quantify momentum drift (should be ≈ machine epsilon for symplectic integrator)
println("Momentum conservation (max deviation from t=0 values):")
@printf("  %-18s  %-12s  %-12s  %-12s\n", "Run", "max|ΔPx|", "max|ΔPy|", "max|ΔLz|")
for (sol, label) in zip(sols, run_labels)
    Px0 = sum(sol.p[1][1:2:end])
    Py0 = sum(sol.p[1][2:2:end])
    Lz0 = sum(sol.q[1][1:2:end].*sol.p[1][2:2:end] .-
               sol.q[1][2:2:end].*sol.p[1][1:2:end])
    dPx = maximum([abs(sum(sol.p[k][1:2:end]) - Px0) for k in eachindex(sol.t)])
    dPy = maximum([abs(sum(sol.p[k][2:2:end]) - Py0) for k in eachindex(sol.t)])
    dLz = maximum([abs(sum(sol.q[k][1:2:end].*sol.p[k][2:2:end] .-
                            sol.q[k][2:2:end].*sol.p[k][1:2:end]) - Lz0)
                   for k in eachindex(sol.t)])
    @printf("  %-18s  %-12.2e  %-12.2e  %-12.2e\n", label, dPx, dPy, dLz)
end

In [ ]:
# Full regularization diagnostics for all four runs
println("Regularization diagnostics:")
@printf("  %-18s  %-11s  %-12s  %-17s  %-13s  %-13s  %-12s\n",
    "Run", "pair_steps", "chain_steps", "max_substeps_used",
    "adpt_steps", "lifted_steps", "min_enc_dist")
for (sol, label) in zip(sols, run_labels)
    rd = sol.regularization
    @printf("  %-18s  %-11d  %-12d  %-17d  %-13d  %-13d  %-12.4e\n",
        label, rd.pair_steps, rd.chain_steps, rd.max_substeps_used,
        rd.adaptive_pair_steps, rd.lifted_pair_steps, rd.min_encounter_distance)
end

## API Reference

| Category | Types / Options | Compute Functions | Plot Functions |
|----------|-----------------|-------------------|----------------|
| **System** | `WeberSystem` | — | — |
| **Problem** | `WeberProblem`, `RegularizationOptions`, `ZollnerOptions` | — | — |
| **Solution** | `WeberSolution` | `solve` | — |
| **Trajectories** | `TrajectoryData` | `compute_trajectory_data` | `plot_trajectories` |
| **Energy** | `EnergyData`, `PairEnergyData`, `EnergyStatistics` | `compute_energy_timeseries` | `plot_energy`, `plot_pair_energy`, `plot_energy_errors` |
| **Forces + Phase** | `PairForceData`, `ForceStatistics`, `PhaseSpaceData` | `compute_pair_force_timeseries` | `plot_pair_forces`, `plot_phase_space` |
| **Momentum** | `MomentumData` | `compute_momentum_timeseries` | `plot_momentum_errors` |
| **Zöllner** | `ZollnerOptions` | *(embedded in above)* | `plot_zollner_energy`, `plot_zollner_force_residual`, `plot_weber_vs_zollner`, `plot_zollner_phase_space` |

### Initial Condition Formula (this notebook)

```julia
PE_abs_0 = Q^2 / R * (2*sqrt(2) - 1)   # |PE₀| at t=0, a=0 reference
v        = sqrt(eta * PE_abs_0 / 2)     # tangential speed for KE/|PE₀| = eta
```

Positions: cardinal points of square of radius R (East, North, West, South)
Momenta: tangential CCW — all ṙᵢⱼ = 0 at t=0 → Weber potential = Coulomb at t=0

---

## Part 2: Static Initial Conditions (η = 0)

Same four-body square geometry and Zöllner parameter sweep `a ∈ {0, 0.02, 0.05, 0.10}`,
but particles are **released from rest** (all initial momenta = 0).

At t = 0 the unlike-sign pairs attract and like-sign pairs repel under pure Coulomb
(Weber potential = Coulomb when ṙᵢⱼ = 0). Regularization is essential here because
the particles fall directly toward each other.

The single change from Part 1: `eta = 0.0` → `v = 0` → `p0_s = zeros(...)`.


In [ ]:
# ── Part 2: redefine ICs with η = 0 (released from rest) ────────────────────
eta_s = 0.0
v_s   = 0.0
p0_s  = zeros(Float64, n_particles * dims)

# Visualise: same square positions, zero velocity arrows
p_ic_s = scatter([q0[2k-1] for k in 1:n_particles], [q0[2k] for k in 1:n_particles];
    markersize = 20, markershape = :circle,
    markercolor = [:firebrick, :steelblue, :firebrick, :steelblue],
    markerstrokewidth = 2, label = "",
    aspect_ratio = :equal,
    xlims = (-1.4, 1.4), ylims = (-1.4, 1.4),
    title  = "Static IC: η = 0 (released from rest)",
    xlabel = "x", ylabel = "y",
    framestyle = :box, grid = true, gridalpha = 0.25,
    dpi = 200, size = (650, 650),
)
for (k, sym) in enumerate(["+Q","−Q","+Q","−Q"])
    annotate!(p_ic_s, q0[2k-1], q0[2k], text(sym, :white, :center, 10, :bold))
    offsets = [(0.18,0.18),(-0.22,0.18),(-0.22,-0.20),(0.18,-0.20)]
    annotate!(p_ic_s, q0[2k-1]+offsets[k][1], q0[2k]+offsets[k][2],
              text("P$k", :black, :center, 9))
end
sq_order = [1,2,3,4,1]
plot!(p_ic_s, [q0[2k-1] for k in sq_order], [q0[2k] for k in sq_order];
    linestyle = :dot, color = :gray, linewidth = 1, label = "")
annotate!(p_ic_s, 0.0, 0.0, text("v = 0 for all", :gray40, :center, 9))
p_ic_s


In [ ]:
# Build and integrate four WeberProblems with static ICs
probs_s = [
    WeberProblem(system, tspan, q0, p0_s;
        masses  = masses,
        charges = charges,
        c       = c,
        dt      = dt,
        regularization = reg_opts,
        zollner = ZollnerOptions(enabled = (a > 0.0), a = a),
    )
    for a in a_values
]

println("Integrating static-IC runs …")
sols_s = [solve(prob) for prob in probs_s]

println("\nIntegration results (static IC):")
@printf("  %-18s  %-9s  %-8s  %-12s  %-12s  %-12s\n",
    "Run", "Retcode", "Points", "pair_steps", "chain_steps", "min_sep")
for (sol, label) in zip(sols_s, run_labels)
    rd = sol.regularization
    @printf("  %-18s  %-9s  %-8d  %-12d  %-12d  %-12.4e\n",
        label, sol.retcode, length(sol.t),
        rd.pair_steps, rd.chain_steps, rd.min_encounter_distance)
end


In [ ]:
# 2D trajectory plots — static IC, all four runs (2×2)
traj_plots_s = []
for (sol, label) in zip(sols_s, run_labels)
    traj = compute_trajectory_data(sol, n_particles, dims; stride=1)
    p = plot_trajectories(traj)
    title!(p, "Trajectories (η=0) — $label")
    push!(traj_plots_s, p)
end

plot(traj_plots_s..., layout=(2,2), size=(2100, 2100),
    plot_title="Static-IC 2D Trajectories — All Four Runs")


In [ ]:
# Overlay: Weber (a=0) vs max Zöllner (a=0.10), static ICs
plot_weber_vs_zollner(sols_s[1], sols_s[4];
    labels = [run_labels[1] * " (η=0)", run_labels[4] * " (η=0)"])


In [ ]:
# Compute energy timeseries for static-IC runs
println("Computing energy timeseries (static IC) …")
energies_s = [compute_energy_timeseries(sol; stride=1) for sol in sols_s]
println("Done.\n")

println("Energy conservation summary (static IC):")
@printf("  %-18s  %-12s  %-14s  %-14s\n",
    "Run", "H₀", "max drift %", "max local err")
for (en, label) in zip(energies_s, run_labels)
    s = en.statistics
    @printf("  %-18s  %-12.6f  %-14.4e  %-14.4e\n",
        label, en.total_energy[1],
        s.global_error_percent_max, s.local_error_max)
end


In [ ]:
# Energy timeseries — static IC, all four runs
plot(
    [plot_energy(en) for en in energies_s]...,
    layout=(2,2), size=(2100, 2100),
    plot_title="Energy Conservation — Static IC, All Four Runs",
)


In [ ]:
# Energy error analysis — static IC
plot(
    [plot_energy_errors(en) for en in energies_s]...,
    layout=(2,2), size=(2100, 2625),
    plot_title="Energy Error Analysis — Static IC, All Four Runs",
)


In [ ]:
# Regularization diagnostics — static IC
println("Regularization diagnostics (static IC):")
@printf("  %-18s  %-11s  %-12s  %-17s  %-13s  %-13s  %-12s\n",
    "Run", "pair_steps", "chain_steps", "max_substeps_used",
    "adpt_steps", "lifted_steps", "min_enc_dist")
for (sol, label) in zip(sols_s, run_labels)
    rd = sol.regularization
    @printf("  %-18s  %-11d  %-12d  %-17d  %-13d  %-13d  %-12.4e\n",
        label, rd.pair_steps, rd.chain_steps, rd.max_substeps_used,
        rd.adaptive_pair_steps, rd.lifted_pair_steps, rd.min_encounter_distance)
end
